In [124]:
import pandas as pd

In [125]:
df = pd.read_csv('data.csv',sep=';')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4898 entries, 0 to 4897
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         4898 non-null   float64
 1   volatile acidity      4898 non-null   float64
 2   citric acid           4898 non-null   float64
 3   residual sugar        4898 non-null   float64
 4   chlorides             4898 non-null   float64
 5   free sulfur dioxide   4898 non-null   float64
 6   total sulfur dioxide  4898 non-null   float64
 7   density               4898 non-null   float64
 8   pH                    4898 non-null   float64
 9   sulphates             4898 non-null   float64
 10  alcohol               4898 non-null   float64
 11  quality               4898 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 459.3 KB


In [126]:
df.sample(10)

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
4368,6.5,0.30,0.28,11.45,0.041,29.0,109.0,0.99418,2.98,0.30,10.9,6
1038,7.1,0.15,0.34,5.30,0.034,33.0,104.0,0.99530,3.37,0.52,9.3,7
4150,7.6,0.48,0.31,9.40,0.046,6.0,194.0,0.99714,3.07,0.61,9.4,5
1241,7.4,0.18,0.42,2.10,0.036,33.0,187.0,0.99380,3.40,0.41,10.6,7
3395,6.6,0.26,0.46,7.80,0.047,48.0,186.0,0.99580,3.20,0.54,9.1,5
4311,7.1,0.23,0.28,1.90,0.046,33.0,103.0,0.98997,3.12,0.31,12.0,5
3240,6.4,0.16,0.42,1.00,0.036,29.0,113.0,0.99080,3.18,0.52,11.0,6
1238,7.1,0.44,0.37,2.70,0.041,35.0,128.0,0.98960,3.07,0.43,13.5,7
552,7.8,0.28,0.34,1.60,0.028,32.0,118.0,0.99010,3.00,0.38,12.1,7
4423,6.2,0.22,0.28,2.20,0.040,24.0,125.0,0.99170,3.19,0.48,10.5,6


In [127]:
df.isna().sum()

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

In [128]:
df.drop_duplicates(inplace=True)

In [129]:
df.duplicated().sum()

np.int64(0)

In [130]:
X = df.drop(columns=['quality'])
Y = df['quality']

In [131]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [132]:
X_train,X_temp,Y_train,Y_temp = train_test_split(X,Y,test_size=0.2,random_state=42)
X_val,X_test,Y_val,Y_test = train_test_split(X_temp,Y_temp,test_size=0.56,random_state=42) 

In [133]:
X_val.shape

(348, 11)

In [134]:
X_test.shape

(445, 11)

In [135]:
X_train.shape

(3168, 11)

In [136]:
ss = StandardScaler()
X_train = ss.fit_transform(X_train)
X_val = ss.transform(X_val)
X_test = ss.transform(X_test)

In [137]:
import tensorflow as tf
from tensorflow.keras.layers import Dense,Input,Dropout
from tensorflow.keras.models import Sequential
import keras_tuner as kt

In [138]:
def build_model(hp):
    nodes = hp.Int('nodes',8,128,step=8)
    dropout_val = hp.Float('dropout',min_value=0.1,max_value=0.8,step=0.1)
    
    model = Sequential()
    model.add(Input(shape=(X_train.shape[1],)))
    for i in range(hp.Int('hidden',min_value=1,max_value=10,step=1)):
        model.add(Dense(nodes,activation='relu'))
        model.add(Dropout(dropout_val))
    model.add(Dense(1,activation='linear'))

    optimizer = hp.Choice('optimizer',['SGD','RMSprop','Adam','AdamW','Adadelta','Adamax','Adagrad'])

    model.compile(optimizer=optimizer,loss='mse',metrics=['mae'])
    return model


In [139]:
tuner = kt.RandomSearch(build_model,objective='val_loss',max_trials=20,directory='All_in_one',project_name='wine_quality',
    overwrite=True)

In [140]:
tuner.search(X_train,Y_train,validation_data=(X_val,Y_val),epochs=10)

Trial 20 Complete [00h 00m 04s]
val_loss: 0.949444055557251

Best val_loss So Far: 0.527962863445282
Total elapsed time: 00h 01m 22s


In [141]:
tuner.get_best_hyperparameters()[0].values

{'nodes': 48, 'dropout': 0.2, 'hidden': 3, 'optimizer': 'RMSprop'}

In [142]:
model = tuner.get_best_models(num_models=1)[0]

f:\Codes\Python\Gen AI\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(store)


In [143]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    )
]
model.fit(X_train,Y_train,validation_data=(X_val,Y_val),epochs=100,batch_size=32,callbacks=callbacks)

Epoch 1/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.8811 - mae: 0.7348 - val_loss: 0.5720 - val_mae: 0.5958
Epoch 2/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.8880 - mae: 0.7430 - val_loss: 0.5456 - val_mae: 0.5821
Epoch 3/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.8728 - mae: 0.7281 - val_loss: 0.5315 - val_mae: 0.5756
Epoch 4/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.8617 - mae: 0.7260 - val_loss: 0.5238 - val_mae: 0.5707
Epoch 5/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.8298 - mae: 0.7147 - val_loss: 0.5752 - val_mae: 0.5937
Epoch 6/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.8057 - mae: 0.7031 - val_loss: 0.5405 - val_mae: 0.5763
Epoch 7/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.8096 - mae: 0.7048 - val_loss: 0.4920 - val_mae: 0.5494
Epoch 8/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.7977 - mae: 0.6971 - val_loss: 0.4980 - val_mae: 0.5540
Epoch 9/100
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.746

In [145]:
test_loss, test_mae = model.evaluate(X_test, Y_test)

print("Test Loss:", test_loss)
print("Test MAE:", test_mae)

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6353 - mae: 0.6046 
Test Loss: 0.6353083848953247
Test MAE: 0.6045987606048584
